# SRAM controller driver

Runs the memory controller IP from the PS: writes 1–256 into SRAM1, starts the controller through
the control/status register, waits for done, reads the 192 packed words back from SRAM2 and
compares them with the course's answer.

1. Copy `sram_controller.bit` and `sram_controller.hwh` from `build/bd/axi/` to the board, next to
   this notebook and `answer_memory.npy`.
2. Run the cells in order. The controller stays done until it is reset, so load the overlay again
   before a second run.

In [ ]:
from pynq import Overlay
import numpy as np
import time

In [ ]:
class PynqTestDriver:
    def __init__(self, bitfile_path):
        self.hw = Overlay(bitfile_path)
        self.csr = self.hw.csr_0.mmio.array
        self.sram1 = self.hw.SRAM1.mmio.array
        self.sram2 = self.hw.SRAM2.mmio.array
        self.result = np.zeros(192)

    def get_output(self, length):
        return self.sram2[0:length]

    def start(self, input_array):
        start_time = time.time()
        self.sram1[0:len(input_array)] = input_array    # 1. input data into SRAM1
        self.csr[1] = 1                                 # 2. start (the register IP clears the bit again)
        while self.csr[0] != 1:                         # 3. wait for done
            pass
        self.result = np.array(self.get_output(192))    # 4. packed result from SRAM2
        end_time = time.time()
        runtime = end_time - start_time
        print(f"Runtime: {runtime*1000:.3f}ms")

    def verify_output(self, answer_path="answer_memory.npy", length=192):
        answer = np.load(answer_path)
        mismatch_count = np.sum(self.result[:length] != answer[:length])
        if mismatch_count == 0:
            print("Verification Passed")
        else:
            print(f"Verification Failed: {mismatch_count} of {length} words differ")

In [ ]:
driver = PynqTestDriver("sram_controller.bit")

In [ ]:
test_input = np.arange(256, dtype=np.int32) + 1
driver.start(test_input)

In [ ]:
driver.verify_output()

In [ ]:
print(driver.csr[1])    # 0: the start bit was cleared after its pulse